# Predicting how fast people reply to my Telegram messages: preparing the data

This notebook turns my private Telegram export into the two tables that the models use. It is the only part that reads message text, so it only runs on my computer. The second notebook (telegram_response_time.ipynb) starts from the two tables and does not need the export, the labels or any message.

1. Load the Telegram export and keep the 100 contacts I talked to most in the last year (by hand)
2. Add my labels and a language group for every chat (by hand)
3. Compute reply times, with and without merging bursts of messages (by hand)
4. Sentiment of every message (VADER library plus a small word list I wrote), and a few anonymised examples
5. Features (by hand) and the two tables model_burst.csv and model_naive.csv

The tables contain one row per unit of my messages with the contact pseudonym (P001, P002, ...), the contact labels, the reply time and the features. They contain no message text, no names and no dates.

Packages: numpy, pandas and vaderSentiment.

In [10]:
# pip install numpy pandas vaderSentiment
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DATA_DIR = Path("data")
LABELS_PATH = Path("labels.csv")

TOP_N = 100
WINDOW_DAYS = 365      # "last year", counted back from the last message in the export
BURST_GAP = 10 * 60    # my messages less than 10 minutes apart form one burst (seconds)
MAX_DELAY = 24 * 3600  # replies slower than 24 hours count as no reply (seconds)
CYRILLIC_LIMIT = 0.5   # chats with at least this share of cyrillic letters are "ru_uk"
SEED = 42

## Part 1: Loading the data

The data comes from Telegram Desktop (Settings, Advanced, Export Telegram data) as JSON, with media switched off and only personal chats selected. The file result.json goes into the data folder.

Each message has a sender, a date and a text. The text is either a string or a list of pieces (plain text mixed with links and formatting), so it is flattened first. Service messages and forwarded messages are dropped, because forwarded text is not something I wrote.

I keep the 100 chats with the most messages (both sides together) in the last 365 days of the export. Only chats where both of us wrote in that window are allowed. For these 100 contacts I keep their whole history. Choosing the most active chats favours people who reply, which I mention as a limitation in the report.

In [11]:
def flatten_text(text):
    """Telegram stores text as a string or as a list of pieces, return one string."""
    if isinstance(text, list):
        return "".join(p if isinstance(p, str) else p.get("text", "") for p in text)
    return str(text)


def load_messages(data_dir):
    """Read all json exports in data_dir, one row per message of the one-on-one chats."""
    chats = []
    for path in sorted(data_dir.glob("**/*.json")):
        with open(path, encoding="utf-8") as f:
            export = json.load(f)
        if "chats" in export:
            chats += export["chats"]["list"]
        elif "messages" in export:
            chats.append(export)

    rows = []
    for chat in chats:
        if chat.get("type") != "personal_chat":
            continue
        for m in chat["messages"]:
            if m.get("type") != "message" or "forwarded_from" in m:
                continue
            rows.append({
                "chat_id": chat["id"],
                "chat_name": chat.get("name") or "",
                "from_id": m.get("from_id"),
                "date": m["date"],
                "unix": m.get("date_unixtime"),
                "text": flatten_text(m.get("text", "")),
                "has_media": int(any(k in m for k in ("photo", "file", "media_type", "sticker_emoji"))),
            })

    df = pd.DataFrame(rows)
    df["dt"] = pd.to_datetime(df["date"])
    # seconds since 1970, used for time differences (older exports have no unix time)
    from_date = (df["dt"] - pd.Timestamp("1970-01-01")).dt.total_seconds()
    df["t"] = pd.to_numeric(df["unix"], errors="coerce").fillna(from_date)
    return df.drop(columns=["date", "unix"])

In [12]:
raw = load_messages(DATA_DIR)

# I am the sender who appears in the most chats
me = raw.groupby("from_id")["chat_id"].nunique().idxmax()
raw["sender"] = np.where(raw["from_id"] == me, "me", "them")

last_year = raw[raw["dt"] > raw["dt"].max() - pd.Timedelta(days=WINDOW_DAYS)]
counts = last_year.groupby(["chat_id", "sender"]).size().unstack(fill_value=0)
counts = counts.reindex(columns=["me", "them"], fill_value=0)
counts["total"] = counts["me"] + counts["them"]
both_wrote = counts[(counts["me"] > 0) & (counts["them"] > 0)]
top = both_wrote.sort_values("total", ascending=False).head(TOP_N)

n_chats = raw["chat_id"].nunique()
raw = raw[raw["chat_id"].isin(top.index)].copy()

# contacts become P001, P002, ... so the names stay out of the analysis
raw["pid"] = ["P%03d" % (i + 1) for i in pd.factorize(raw["chat_id"])[0]]
contacts = raw.groupby("pid").agg(chat_id=("chat_id", "first"), chat_name=("chat_name", "first")).reset_index()
contacts["n_last_year"] = contacts["chat_id"].map(top["total"])
raw = raw.drop(columns="chat_name")

print(f"kept {len(top)} of {n_chats} chats, {len(raw):,} messages ({(raw['sender'] == 'me').mean():.0%} from me), "
      f"{raw['dt'].min():%Y-%m-%d} to {raw['dt'].max():%Y-%m-%d}")
print(f"messages in the last year per kept chat: {top['total'].min()} to {top['total'].max()}")

kept 100 of 601 chats, 109,450 messages (53% from me), 2019-03-25 to 2026-10-07
messages in the last year per kept chat: 14 to 2519


## Part 2: My labels and language groups

The first time this runs it writes labels.csv with one row per kept contact (most active first) and stops on purpose. I fill in:

* relationship: a short label such as classmate, fr, close_fr or family
* sex: F or M
* age: a number
* lang_override (optional): en or ru_uk, only if the automatic language guess is wrong

Rows with a missing relationship, sex or age are dropped. labels.csv contains real names, so it stays on my computer.

The language group comes from the share of cyrillic letters in the chat, counting both sides. At least half cyrillic means ru_uk (Russian or Ukrainian, I do not separate them), anything else means en.

In [13]:
def cyrillic_share(texts):
    """Share of the letters in a chat that are cyrillic."""
    letters = re.findall(r"[^\W\d_]", " ".join(texts))
    if not letters:
        return 0.0
    return sum("\u0400" <= ch <= "\u04FF" for ch in letters) / len(letters)


if not LABELS_PATH.exists():
    template = contacts.sort_values("n_last_year", ascending=False)[["chat_id", "chat_name", "n_last_year"]]
    template = template.assign(relationship="", sex="", age=np.nan, lang_override="")
    template.to_csv(LABELS_PATH, index=False)
    raise RuntimeError(f"Wrote {LABELS_PATH}. Fill in relationship, sex and age, then run again from the top.")

labels = pd.read_csv(LABELS_PATH).dropna(subset=["relationship", "sex", "age"])
labels["relationship"] = labels["relationship"].astype(str).str.strip().str.lower().str.replace(" ", "_")
labels["sex"] = labels["sex"].astype(str).str.strip().str.upper().str[0]

people = labels.merge(contacts[["chat_id", "pid"]], on="chat_id")
people["cyr_share"] = people["pid"].map(raw.groupby("pid")["text"].apply(cyrillic_share))
people["lang_group"] = np.where(people["cyr_share"] >= CYRILLIC_LIMIT, "ru_uk", "en")
fixed = people["lang_override"].isin(["en", "ru_uk"])
people.loc[fixed, "lang_group"] = people.loc[fixed, "lang_override"]

print(len(people), "labelled contacts")
display(pd.crosstab(people["lang_group"], people["relationship"]))

99 labelled contacts


relationship,classmate,close_fr,family,fr
lang_group,,,,
en,58,6,0,9
ru_uk,7,5,7,7


## Part 3: Reply times

For a unit $u$ of my messages (one message, or one burst) let $t_u$ be the time of its last message. Then

$$r_u=\min\{t_j:\ \text{sender}_j=\text{them},\ t_j>t_u\},\qquad \Delta_u=r_u-t_u,\qquad y_u=\log(1+\Delta_u).$$

I build two versions of the data. In the naive version every message I send is its own unit. In the burst version, messages from me that are less than 10 minutes apart, with nothing from the other person in between, are merged into one unit. The texts are joined and the clock starts at the last message of the burst, because that is when the other person can start replying to the whole thing.

Units with no later reply, or with a reply after more than 24 hours, are dropped. This removes the slowest replies, so the data leans towards fast ones.

Two more features need the order of the chat, so they are recorded here: prev_gap (seconds since the previous message from either side) and prev_from_them (1 if the previous message was theirs, so I am replying to them).

In [14]:
def response_times(chat, burst=True):
    """Reply times for one contact, one row per unit of my messages.

    A unit is a single message (burst=False) or a burst of my messages sent less
    than BURST_GAP seconds apart (burst=True). The reply time runs from the last
    message of the unit to the next message from them. Units that were never
    answered, or answered after more than MAX_DELAY, are dropped.
    Returns the table and the number of units before dropping.
    """
    chat = chat.sort_values("t", kind="stable").reset_index(drop=True)
    t = chat["t"].to_numpy(float)
    who = chat["sender"].to_numpy()
    n = len(chat)

    # time of the first message from them after each message, found by scanning backwards
    next_them = np.full(n, np.nan)
    upcoming = np.nan
    for i in range(n - 1, -1, -1):
        next_them[i] = upcoming
        if who[i] == "them":
            upcoming = t[i]

    units, current = [], []
    for i in range(n):
        if who[i] == "me":
            if burst and current and t[i] - t[current[-1]] < BURST_GAP:
                current.append(i)
            else:
                if current:
                    units.append(current)
                current = [i]
        elif current:
            units.append(current)
            current = []
    if current:
        units.append(current)

    rows = []
    for unit in units:
        first, last = unit[0], unit[-1]
        delay = next_them[last] - t[last]
        if np.isnan(delay) or delay > MAX_DELAY:
            continue
        rows.append({
            "delay": delay,
            "send_dt": chat["dt"].iloc[last],
            "n_msgs": len(unit),
            "text": " ".join(chat["text"].iloc[unit]),
            "has_media": int(chat["has_media"].iloc[unit].max()),
            "prev_gap": t[first] - t[first - 1] if first > 0 else np.nan,
            "prev_from_them": int(first > 0 and who[first - 1] == "them"),
        })
    return pd.DataFrame(rows), len(units)


def build_dataset(raw, burst):
    """Reply-time table for all contacts together and the number of units before dropping."""
    parts, n_units = [], 0
    for pid, chat in raw.groupby("pid"):
        table, n = response_times(chat, burst)
        n_units += n
        if len(table) > 0:
            parts.append(table.assign(pid=pid))
    data = pd.concat(parts, ignore_index=True)
    name = "burst" if burst else "naive"
    print(f"{name}: {len(data)} of {n_units} units kept ({1 - len(data) / n_units:.1%} dropped)")
    return data


df_burst = build_dataset(raw, burst=True)
df_naive = build_dataset(raw, burst=False)

# merging bursts can only reduce the number of units, and every delay must be in (0, 24 h]
assert len(df_burst) <= len(df_naive)
assert df_burst["delay"].between(0, MAX_DELAY).all() and df_naive["delay"].between(0, MAX_DELAY).all()

burst: 23622 of 25187 units kept (6.2% dropped)
naive: 54552 of 58428 units kept (6.6% dropped)


## Part 4: Sentiment of my messages

Every message gets one sentiment number between -1 and 1. I use VADER, a rule based scorer that runs on my own computer, so no message leaves it. VADER knows English words, emoji and punctuation. It does not know Russian or Ukrainian, so I add a small list of positive and negative word stems for those languages, with simple negation (a "не" in front of a word flips its sign). The two scores are added and clipped to $[-1, 1]$. The report explains how VADER works.

This is a crude scorer. It misses slang, sarcasm and every word that is not in its lists, and it gives exactly 0 to anything it does not recognise. Many chat messages are neutral anyway, so many zeros are expected.

In [15]:
STEMS_POS = ("спасиб", "дяку", "благодар", "круто", "крутий", "крутой", "класс", "класн", "супер", "отличн",
             "відмінн", "чудов", "чудес", "прекрасн", "замечательн", "шикарн", "здорово", "приятн", "приємн",
             "весел", "смешн", "смішн", "ахах", "хаха", "обожа", "обожн", "люблю", "кохаю", "счастлив", "щасли",
             "молодец", "молодц", "красив", "гарно", "гарна", "гарний", "поздравл", "вітаю", "привітаю", "удач",
             "успех", "успіх", "обним", "обійма", "ржу", "ржак", "кайф")
EXACT_POS = {"рад", "рада", "рады", "радий", "раді", "лол", "целую", "цілую", "огонь"}
STEMS_NEG = ("плох", "поган", "ужас", "жах", "ненавиж", "ненавид", "ненавис", "злюсь", "злюс", "злой", "злий",
             "бесит", "бісить", "бісит", "дратує", "раздража", "груст", "сумн", "жаль", "шкода", "обидн",
             "разочар", "розчар", "отстой", "кошмар", "проблем", "устал", "втомив", "втомл", "боюсь", "боюс",
             "страшн", "тревож", "тривож", "болит", "болить", "плачу", "плакал", "дурак", "дурн", "идиот",
             "ідіот", "стресс", "стрес", "паршив", "мерзк", "отврат", "огидн")
NEGATORS = {"не", "ни", "ні", "нет", "без", "нема", "немає"}

# VADER reads these words as sentiment, but in chat they are just filler
FILLERS = re.compile(r"\b(ok|okay|yes|no|please|well|sure)\b", re.IGNORECASE)
vader = SentimentIntensityAnalyzer()


def ru_uk_score(text):
    """Score Russian and Ukrainian text by counting positive and negative word stems.

    A negator right before a word flips its sign. The count is squashed into (-1, 1)
    in the same way VADER squashes its sum.
    """
    words = re.findall(r"[^\W\d_]+", text.lower())
    total = 0
    for i, w in enumerate(words):
        if w in EXACT_POS or w.startswith(STEMS_POS):
            sign = 1
        elif w.startswith(STEMS_NEG):
            sign = -1
        else:
            continue
        if i > 0 and words[i - 1] in NEGATORS:
            sign = -sign
        total += sign
    return total / np.sqrt(total ** 2 + 15)


def sentiment_score(text):
    """VADER score for the English part plus the stem score, clipped to [-1, 1]."""
    english = vader.polarity_scores(FILLERS.sub(" ", text))["compound"]
    return float(np.clip(english + ru_uk_score(text), -1, 1))


texts = pd.concat([df_burst["text"], df_naive["text"]]).unique()
sentiment = {t: sentiment_score(t) for t in texts}
share_zero = np.mean([v == 0 for v in sentiment.values()])
print(f"scored {len(sentiment)} distinct texts, {share_zero:.0%} are exactly 0")

scored 60246 distinct texts, 55% are exactly 0


### A few anonymised examples of the scoring

The messages themselves stay private, but to show how the scorer behaves I save a small sample of my own short messages with their scores. A message is only a candidate if it has no digits, no link, no @handle and no capitalised word apart from the first one and the word I, so names are unlikely to get in. This filter is not perfect (a name written in lower case gets through), so I read every row before sharing the file.

For every language group I take up to 5 negative, 5 neutral and 5 positive messages. The columns show the two parts of the score and their sum, clipped to $[-1, 1]$.

In [16]:
def may_identify(text):
    """True if the text could contain a name, number, link or handle."""
    if re.search(r"https?://|www\.|@|\d", text):
        return True
    words = text.split()
    for i, w in enumerate(words):
        core = w.strip(".,!?;:()\"'").replace("\u2019", "'")
        if core and core[0].isupper() and not (core == "I" or core.startswith("I'")):
            if i > 0 or w[-1] in ",:;-":
                return True
    return False


pool = df_burst.merge(people[["pid", "lang_group"]], on="pid")
pool = pool[pool["text"].str.len().between(10, 80)].drop_duplicates("text")
pool = pool[~pool["text"].map(may_identify)].copy()
pool["vader_part"] = pool["text"].map(lambda t: vader.polarity_scores(FILLERS.sub(" ", t))["compound"])
pool["word_list_part"] = pool["text"].map(ru_uk_score)
pool["sentiment"] = pool["text"].map(sentiment)
pool["kind"] = np.sign(pool["sentiment"]).map({-1.0: "negative", 0.0: "neutral", 1.0: "positive"})

parts = [g.sample(min(2, len(g)), random_state=SEED) for _, g in pool.groupby(["lang_group", "kind"])]
examples = pd.concat(parts).sort_values(["lang_group", "sentiment"])
examples = examples[["lang_group", "text", "vader_part", "word_list_part", "sentiment"]].round(3)
examples.to_csv("sentiment_examples.csv", index=False)
display(examples)

,lang_group,text,vader_part,word_list_part,sentiment
18538,en,We are cooked 😭,-0.477,0.00,-0.477
7195,en,Nah it accumulates over time,-0.103,0.00,-0.103
4668,en,Ahahahaha),0.000,0.00,0.000
22684,en,when do u have it bro i assume tomorrow morning,0.000,0.00,0.000
8372,en,'good for you',0.440,0.00,0.440
13031,en,Brilliant!,0.624,0.00,0.624
11867,ru_uk,cardio kills gains,-0.273,0.00,-0.273
19545,ru_uk,ненавиджу все їхнє,0.000,-0.25,-0.250
19528,ru_uk,А взагалі про що в тебе аплікашка?,0.000,0.00,0.000
3318,ru_uk,Якщо в фінансовому плані лоан уже порахований ...,0.000,0.00,0.000


## Part 5: Features

Message features: sentiment, the log of the message length, a question mark flag, an exclamation mark flag, a media flag, and the log of the burst length (always 0 in the naive version). The time of day is the number of hours since midnight (a number from 0 to 24, so 07:30 is 7.5 and 23:00 is 23), plus a weekend flag. Context features: the log of the gap since the previous message and whether the previous message was from them. Contact features: relationship, sex and language group and age.

The target is $y=\log(1+\text{seconds})$.

In [17]:
MESSAGE_FEATURES = ["sentiment", "log_len", "has_question", "has_exclaim", "has_media", "hours_since_midnight",
                    "is_weekend", "log_prev_gap", "prev_from_them", "log_n_msgs"]


def add_features(data):
    """Attach the contact labels and build all features and the target y = log(1 + seconds)."""
    d = data.merge(people[["pid", "relationship", "sex", "age", "lang_group"]], on="pid")
    d["y"] = np.log1p(d["delay"])
    d["sentiment"] = d["text"].map(sentiment)
    d["log_len"] = np.log1p(d["text"].str.len())
    d["has_question"] = d["text"].str.contains("?", regex=False).astype(int)
    d["has_exclaim"] = d["text"].str.contains("!", regex=False).astype(int)
    d["hours_since_midnight"] = d["send_dt"].dt.hour + d["send_dt"].dt.minute / 60
    d["is_weekend"] = (d["send_dt"].dt.dayofweek >= 5).astype(int)
    d["log_prev_gap"] = np.log1p(d["prev_gap"])
    d["log_prev_gap"] = d["log_prev_gap"].fillna(d["log_prev_gap"].median())
    d["log_n_msgs"] = np.log(d["n_msgs"])
    return d


D_burst = add_features(df_burst)
D_naive = add_features(df_naive)
print(D_burst.shape, D_naive.shape)

(23604, 21) (54515, 21)


### Saving the two tables

The tables keep only what the models need: the contact pseudonym, the contact labels, the reply time in seconds, its logarithm and the features. The message text, the sending time and the chat ids are left out. These two files are what the second notebook reads.

In [18]:
SHARED_COLUMNS = ["pid", "relationship", "sex", "age", "lang_group", "delay", "y"] + MESSAGE_FEATURES
D_burst[SHARED_COLUMNS].to_csv("model_burst.csv", index=False)
D_naive[SHARED_COLUMNS].to_csv("model_naive.csv", index=False)
display(D_burst[SHARED_COLUMNS].head())

,pid,relationship,sex,age,lang_group,delay,y,sentiment,log_len,has_question,has_exclaim,has_media,hours_since_midnight,is_weekend,log_prev_gap,prev_from_them,log_n_msgs
0,P001,family,F,45,ru_uk,7.0,2.079442,0.0,3.931826,1,0,0,6.266667,0,8.787831,1,1.098612
1,P001,family,F,45,ru_uk,7.0,2.079442,0.0,1.386294,0,0,0,6.283333,0,3.737670,1,0.000000
2,P001,family,F,45,ru_uk,47.0,3.871201,0.0,3.295837,0,0,0,6.316667,0,4.644391,1,0.000000
3,P001,family,F,45,ru_uk,33.0,3.526361,0.0,3.433987,0,0,0,0.683333,0,3.044522,1,0.000000
4,P001,family,F,45,ru_uk,51.0,3.951244,0.0,3.367296,0,0,0,2.433333,0,12.770716,0,0.000000
